# Análisis de Emisiones CO2 - Modelo FCR

Este notebook demuestra el cálculo de emisiones de CO2 basado en el modelo **Fuel Consumption Rate (FCR)** de Xiao et al. (2012), ajustado con los pesos reales de la operación.

In [4]:
import pandas as pd

# =============================================================================
# CONFIGURACIÓN (Basada en src/config.py)
# =============================================================================
DEFAULT_CO2_PER_KM = 1.57        # kg CO2/km a plena carga
DEFAULT_ALPHA_FCR = 0.5          # Ratio de consumo vacío/cargado
PAPER_LOAD_KG = 25000           # Carga de bobinas PP -> CP (kg)
PALLET_WEIGHT_KG = 145           # Peso promedio por pallet (kg)
VEHICLE_MAX_LOAD_KG = 25000     # Capacidad máxima del vehículo para el cálculo del ratio de carga

class FCREmissionEstimator:
    def __init__(self, fcr_loaded, alpha=0.5):
        self.fcr_loaded = fcr_loaded
        self.alpha = alpha
    
    @property
    def fcr_empty(self):
        return self.fcr_loaded * self.alpha

    def co2_partial_load(self, distance_km, current_load, max_load):
        fcr_arc = self.fcr_empty + (
            (self.fcr_loaded - self.fcr_empty) / max_load
        ) * current_load
        return distance_km * fcr_arc

estimator = FCREmissionEstimator(DEFAULT_CO2_PER_KM, DEFAULT_ALPHA_FCR)

### Escenario: 1 Camión por Ruta

Definimos una secuencia más realista de entrega:

In [5]:
pallets_totales = 32
ruta = [
    {"Tramo": "1. Depósito -> Planta (Papel)", "KM": 310, "Tipo": "Papel (25t)", "Carga_kg": PAPER_LOAD_KG},
    {"Tramo": "2. Planta -> Cliente 1", "KM": 45, "Tipo": "Reparto", "Carga_kg": pallets_totales * PALLET_WEIGHT_KG},
    {"Tramo": "3. Cliente 1 -> Cliente 2", "KM": 60, "Tipo": "Reparto", "Carga_kg": (pallets_totales - 12) * PALLET_WEIGHT_KG},
    {"Tramo": "4. Cliente 2 -> Depósito (Vacío)", "KM": 290, "Tipo": "Retorno", "Carga_kg": 0}
]

res = []
for t in ruta:
    co2 = estimator.co2_partial_load(t["KM"], t["Carga_kg"], VEHICLE_MAX_LOAD_KG)
    res.append({
        "Tramo": t["Tramo"], 
        "KM": t["KM"], 
        "Tipo": t["Tipo"],
        "Carga (kg)": t["Carga_kg"], 
        "CO2 (kg)": round(co2, 2),
        "KPI (kg/km)": round(co2/t["KM"], 3)
    })

df = pd.DataFrame(res)
df

,Tramo,KM,Tipo,Carga (kg),CO2 (kg),KPI (kg/km)
0,1. Depósito -> Planta (Papel),310,Papel (25t),25000,486.70,1.570
1,2. Planta -> Cliente 1,45,Reparto,4640,41.88,0.931
2,3. Cliente 1 -> Cliente 2,60,Reparto,2900,52.56,0.876
3,4. Cliente 2 -> Depósito (Vacío),290,Retorno,0,227.65,0.785


### Resumen de la Ruta

In [6]:
print(f"Emisiones Totales: {df['CO2 (kg)'].sum():.2f} kg CO2")
print(f"Distancia Total: {df['KM'].sum()} km")

Emisiones Totales: 808.79 kg CO2
Distancia Total: 705 km
